# 🩺 Diabetes Prediction using K-Nearest Neighbours (KNN)

**Dataset:** Pima Indians Diabetes Dataset  
**Goal:** Predict whether a patient is diabetic (1) or not (0)  
**Algorithm:** K-Nearest Neighbours (KNN)  

---

## 📋 Pipeline
1. Load & understand the data
2. Exploratory Data Analysis (EDA)
3. Handle missing values
4. Preprocess (scale features)
5. Find optimal K via cross-validation
6. Train & evaluate the model
7. Compare KNN vs other models

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✅')

## Step 2 — Load the Dataset

> The dataset has **768 rows** and **9 columns** (8 features + 1 target).  
> Target: `Outcome` — 0 = No Diabetes, 1 = Diabetes

In [ ]:
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
COLS = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigree','Age','Outcome']

df = pd.read_csv(URL, names=COLS)
print('Shape:', df.shape)
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:', df.isnull().sum().sum())
print('\nClass Distribution:')
print(df['Outcome'].value_counts().rename({0:'No Diabetes', 1:'Diabetes'}))

In [ ]:
df.describe().round(2)

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Outcome'].value_counts()
bars = ax.bar(['No Diabetes', 'Diabetes'], counts,
              color=['steelblue', 'salmon'], edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5, str(count),
            ha='center', fontweight='bold')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Zeros in medical columns = actually missing values!
ZERO_COLS = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
print('⚠️  Zero counts (biologically impossible = missing):')
print((df[ZERO_COLS] == 0).sum())

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 Glucose & BMI have the highest correlation with Outcome

In [ ]:
# Feature distributions by class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feat in zip(axes.flatten(), COLS[:-1]):
    for label, color, name in zip([0,1], ['steelblue','salmon'],
                                   ['No Diabetes','Diabetes']):
        df[df['Outcome']==label][feat].hist(
            bins=20, alpha=0.6, color=color, label=name,
            edgecolor='white', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('')
axes[0][0].legend()
plt.suptitle('Feature Distributions by Class', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing

### Two things to fix:
1. **Replace zero values** in medical columns with the column median
2. **Scale features** — KNN uses distance, so large-range features dominate without scaling

In [ ]:
# Fix zeros → replace with column median
for col in ZERO_COLS:
    median_val = df[col].median()
    zeros = (df[col] == 0).sum()
    df[col] = df[col].replace(0, median_val)
    print(f'{col:<20} → fixed {zeros} zeros (median={median_val:.1f})')

print('\nZeros remaining:', (df[ZERO_COLS] == 0).sum().sum())

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

# Scale — learn ONLY from training data
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)        # transform only (never fit on test!)

print('Scaling done ✅ (fit only on train set)')

## Step 5 — Find Optimal K

> **K=1** → model memorises every training point → overfits  
> **K=too large** → model ignores local patterns → underfits  
> We use **5-fold cross-validation** to find the sweet spot.

In [ ]:
K_RANGE = range(1, 26)
cv_scores, train_scores = [], []

for k in K_RANGE:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv  = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring='accuracy').mean()
    knn.fit(X_train_sc, y_train)
    cv_scores.append(cv)
    train_scores.append(knn.score(X_train_sc, y_train))

best_k = K_RANGE[np.argmax(cv_scores)]
print(f'✅ Best K = {best_k}  |  CV Accuracy = {max(cv_scores):.4f}')

# Plot
plt.figure(figsize=(11, 5))
plt.plot(K_RANGE, train_scores, 'o-', color='coral', label='Train Accuracy', markersize=4)
plt.plot(K_RANGE, cv_scores,   's-', color='steelblue', label='CV Accuracy (5-fold)', markersize=4)
plt.axvline(best_k, linestyle='--', color='green', alpha=0.7, label=f'Best K={best_k}')
plt.xlabel('K (Number of Neighbours)')
plt.ylabel('Accuracy')
plt.title('Finding Optimal K — Overfitting Check', fontweight='bold')
plt.xticks(list(K_RANGE))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6 — Train & Evaluate Final KNN Model

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_sc, y_train)

y_pred = knn_model.predict(X_test_sc)
y_prob = knn_model.predict_proba(X_test_sc)[:, 1]

knn_acc = accuracy_score(y_test, y_pred)
knn_auc = roc_auc_score(y_test, y_prob)

print(f'Test Accuracy : {knn_acc:.4f}')
print(f'ROC-AUC Score : {knn_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Diabetes','Diabetes']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Diabetes','Diabetes'],
    cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — KNN (K={best_k})', fontweight='bold')
plt.tight_layout()
plt.show()

# Reading the matrix:
# Top-left  = True Negatives  (correctly predicted No Diabetes)
# Top-right = False Positives (predicted Diabetes, actually No Diabetes)
# Bot-left  = False Negatives (predicted No Diabetes, actually Diabetes) ← most dangerous!
# Bot-right = True Positives  (correctly predicted Diabetes)

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'KNN (AUC={knn_auc:.3f})')
plt.fill_between(fpr, tpr, alpha=0.08, color='steelblue')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — KNN', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7 — Scaling Impact Demo

> **Why does scaling matter for KNN?**  
> KNN calculates Euclidean distance: √((x₁−y₁)² + (x₂−y₂)² + ...)  
> If `Insulin` ranges 0–800 and `BMI` ranges 15–65, a 1-unit difference in Insulin completely drowns out a 1-unit difference in BMI. Scaling fixes this.

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=best_k)
knn_raw.fit(X_train, y_train)

acc_raw    = knn_raw.score(X_test, y_test)
acc_scaled = knn_model.score(X_test_sc, y_test)

print(f'Accuracy WITHOUT scaling : {acc_raw:.4f}')
print(f'Accuracy WITH scaling    : {acc_scaled:.4f}')
print(f'Improvement              : +{(acc_scaled - acc_raw)*100:.1f}%')

# Bar chart
plt.figure(figsize=(6, 4))
bars = plt.bar(['Without Scaling', 'With Scaling'],
               [acc_raw, acc_scaled],
               color=['salmon', 'steelblue'], edgecolor='white', width=0.4)
for bar, val in zip(bars, [acc_raw, acc_scaled]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.002, f'{val:.4f}',
             ha='center', fontweight='bold')
plt.title('Scaling Impact on KNN Accuracy', fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim(0.65, 0.90)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 — Model Comparison

Let's compare KNN against **Logistic Regression** and **Decision Tree** on the same dataset.

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
lr_prob = lr.predict_proba(X_test_sc)[:, 1]
lr_acc  = accuracy_score(y_test, lr_pred)
lr_auc  = roc_auc_score(y_test, lr_prob)

# Train Decision Tree (no scaling needed)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]
dt_acc  = accuracy_score(y_test, dt_pred)
dt_auc  = roc_auc_score(y_test, dt_prob)

# Summary table
results = pd.DataFrame({
    'Model'   : ['KNN', 'Logistic Regression', 'Decision Tree'],
    'Accuracy': [knn_acc, lr_acc, dt_acc],
    'ROC-AUC' : [knn_auc, lr_auc, dt_auc],
    'Needs Scaling': ['Yes', 'Yes', 'No']
})
results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results[['Accuracy','ROC-AUC']] = results[['Accuracy','ROC-AUC']].round(4)
results

In [ ]:
# Side-by-side bar chart
models  = ['KNN', 'Logistic\nRegression', 'Decision\nTree']
accs    = [knn_acc, lr_acc, dt_acc]
aucs    = [knn_auc, lr_auc, dt_auc]
colors  = ['steelblue', 'mediumseagreen', 'coral']
x       = np.arange(len(models))
width   = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, accs, width, color=colors,
            alpha=0.9, label='Accuracy', edgecolor='white')
b2 = ax.bar(x + width/2, aucs, width, color=colors,
            alpha=0.5, label='ROC-AUC', edgecolor='white', hatch='//')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=10, fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0.60, 1.0)
ax.set_title('Model Comparison — Accuracy & ROC-AUC', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# All 3 ROC curves on one plot
fig, ax = plt.subplots(figsize=(8, 6))
for name, prob, color in zip(
        ['KNN', 'Logistic Regression', 'Decision Tree'],
        [y_prob, lr_prob, dt_prob],
        ['steelblue', 'mediumseagreen', 'coral']):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📌 Summary & Key Takeaways

| Concept | What we learned |
|---|---|
| Zero values | Biologically impossible → treated as missing → replaced with median |
| Scaling | KNN accuracy improves significantly with StandardScaler |
| Optimal K | Found using 5-fold cross-validation |
| Overfitting | K=1 gives 100% train accuracy but poor test — avoid! |
| Best model | Logistic Regression edges out KNN on this dataset |

### 🔑 Golden Rules
- **Always scale** before using KNN
- **Never fit the scaler on test data** — only `transform()`
- Use **cross-validation** to tune K, not just test set accuracy
- **ROC-AUC** is more reliable than accuracy for imbalanced datasets

---
*Project by: Your Name | Dataset: Pima Indians Diabetes*